### Step 1: Generate k-term subsets that have an Arithmetic Progression (AP)

In [1]:
# a, a + d, a + 2d, ..., a + (k-1)d
# thus, a + (k-1)d <= N

from itertools import combinations

def get_subsets(S: range, k: int) -> list:
    return list(combinations(S, k))

def get_arithmetic_progression_subsets(N: int, k: int) -> list:
    progressions = []

    d_max = (N - 1) // (k - 1)  #floor division to get the maximum possible common difference

    for d in range(1, d_max + 1):
        a_max = N - (k - 1) * d

        for a in range(1, a_max + 1):
            progression = tuple(a + i * d for i in range(k))
            progressions.append(progression)

    return progressions

def get_new_arithmetic_progression_subsets(N: int, k: int) -> list:
    progressions = []

    d_max = (N - 1) // (k - 1)

    for d in range(1, d_max + 1):
        a = N - (k - 1) * d
        progression = tuple(a + i * d for i in range(k))
        progressions.append(progression)

    return progressions

def is_ap_free(candidate: tuple, progressions: list) -> bool:
    candidate = set(candidate)
    for progression in progressions:
        if set(progression).issubset(candidate):
            return False

    return True


def r(N: int, k: int, progressions: list,last_size: int = 0) -> tuple:
    S = range(1, N + 1)

    if last_size > 0:
        size = last_size + 1

        for candidate in combinations(S, size):
            if is_ap_free(candidate, progressions):
                return size, candidate

        return last_size, None

    else:
        for size in range(N, 0, -1):
            for candidate in combinations(S, size):
                if is_ap_free(candidate, progressions):
                    return size, candidate


def generate_AP_free_candidates(S, size, k):
    S = sorted(S)

    def creates_kAP(current_set, x):
        """
        Return True if adding x completes a k-term
        arithmetic progression.
        """

        # x will be the final/largest term:
        #
        # x - (k-1)d, ..., x - 2d, x - d, x

        d_max = (x - S[0]) // (k - 1)

        for d in range(1, d_max + 1):

            # Check whether all previous k-1 terms
            # are already in the candidate.
            if all(
                x - i * d in current_set
                for i in range(1, k)
            ):
                return True

        return False


    def backtrack(start, current, current_set):

        # We successfully built a candidate
        if len(current) == size:
            yield tuple(current)
            return

        # Prune if there aren't enough numbers left
        # to reach the requested candidate size.
        remaining_needed = size - len(current)
        remaining_available = len(S) - start

        if remaining_available < remaining_needed:
            return

        for i in range(start, len(S)):
            x = S[i]

            # Would adding x complete a forbidden k-AP?
            if creates_kAP(current_set, x):
                continue

            # Choose x
            current.append(x)
            current_set.add(x)

            # Continue building this branch
            yield from backtrack(
                i + 1,
                current,
                current_set
            )

            # Undo x and try the next possibility
            current.pop()
            current_set.remove(x)


    yield from backtrack(
        0,
        [],
        set()
    )



In [2]:
N = 5
k = 3

cases = 3
results = []

print(f"{'N':>3} | {'r_k(N)':>6}")
print("-" * 12)

max_size = 0

progressions = get_arithmetic_progression_subsets(N, k)

for i in range(N, N + cases):
    if i > N:
        progressions += get_new_arithmetic_progression_subsets(i, k)
    max_size, candidate = r(i, k, progressions,max_size)
    print(f"{i:>3} | {max_size:>6}")

    # Save point for Desmos
    results.append((i, max_size))



""" 

max_size, candidate = r(N,k)
print(f"\nLargest {k}-AP-free subset of {{1, 2, ..., {N}}} has size {max_size}: {candidate}")


ap_subsets = get_arithmetic_progression_subsets(N, k)

combination = (1, 2, 3, 5)
combination2 = (1, 2, 4, 5)

print(f"{combination} is AP-free: {is_ap_free(combination, ap_subsets)}")
print(f"{combination2} is AP-free: {is_ap_free(combination2, ap_subsets)}")

ap_subsets = get_arithmetic_progression_subsets(N, k)
print(f"Count: {len(ap_subsets)}")
print(ap_subsets)


subsets = get_subsets(range(1, N + 1), k)
print(f"C({N},{k}) = {len(subsets)}")
print(subsets) """



  N | r_k(N)
------------
  5 |      4
  6 |      4
  7 |      4


' \n\nmax_size, candidate = r(N,k)\nprint(f"\nLargest {k}-AP-free subset of {{1, 2, ..., {N}}} has size {max_size}: {candidate}")\n\n\nap_subsets = get_arithmetic_progression_subsets(N, k)\n\ncombination = (1, 2, 3, 5)\ncombination2 = (1, 2, 4, 5)\n\nprint(f"{combination} is AP-free: {is_ap_free(combination, ap_subsets)}")\nprint(f"{combination2} is AP-free: {is_ap_free(combination2, ap_subsets)}")\n\nap_subsets = get_arithmetic_progression_subsets(N, k)\nprint(f"Count: {len(ap_subsets)}")\nprint(ap_subsets)\n\n\nsubsets = get_subsets(range(1, N + 1), k)\nprint(f"C({N},{k}) = {len(subsets)}")\nprint(subsets) '

In [3]:
from collections import defaultdict
from time import perf_counter

kap_cache = {}

def creates_kAP(current_set: set, x: int, k: int, cache_stats:dict, min_value: int = 1) -> bool:
    """
    Return True if adding x completes a k-term arithmetic progression.

    Assumes values are added in increasing order, so x is the largest
    term of any newly created arithmetic progression.
    """
    key = (frozenset(current_set), x)
    if key in kap_cache:
        cache_stats["cache_hits"] += 1
        return kap_cache[key]

    cache_stats["cache_misses"] += 1
    d_max = (x - min_value) // (k - 1)

    for d in range(1, d_max + 1):
        if all(
            x - i * d in current_set
            for i in range(1, k)
        ):
            kap_cache[key] = True
            return True

    kap_cache[key] = False
    return False


def make_depth_stats():
    """
    Create a nested dictionary for collecting statistics by search depth.
    """

    return defaultdict(
        lambda: {
            "nodes": 0,
            "cache_hits": 0,
            "cache_misses": 0,
            "ap_prunes": 0,
            "lookahead_prunes": 0,
            "remain_prunes": 0,
            "choices_tried": 0,
            "choices_survived": 0,
            "candidates_yielded": 0
        }
    )


def generate_AP_free_candidates(
    S,
    size: int,
    k: int,
    stats: dict,
    depth_stats
):
    """
    Generate AP-free candidates of the requested size.

    Uses:
      1. short-size pruning
      2. immediate AP pruning
      3. one-step look-ahead pruning
    """

    S = sorted(S)

    if not S:
        return

    min_value = S[0]

    def backtrack(start: int, current: list, current_set: set):

        depth = len(current)

        stats["nodes"] += 1
        depth_stats[depth]["nodes"] += 1

        # ---------------------------------------------
        # SUCCESS
        # ---------------------------------------------

        if len(current) == size:

            stats["candidates_yielded"] += 1
            depth_stats[depth]["candidates_yielded"] += 1

            yield tuple(current)
            return

        # ---------------------------------------------
        # SHORT-SIZE PRUNE
        # ---------------------------------------------

        remaining_needed = size - len(current)
        remaining_available = len(S) - start

        if remaining_available < remaining_needed:

            stats["insufficient_remaining_prunes"] += 1
            depth_stats[depth]["remain_prunes"] += 1

            return

        # ---------------------------------------------
        # TRY POSSIBLE NEXT VALUES
        # ---------------------------------------------

        for i in range(start, len(S)):

            x = S[i]

            stats["choices_tried"] += 1
            depth_stats[depth]["choices_tried"] += 1

            # -----------------------------------------
            # IMMEDIATE AP PRUNE
            # -----------------------------------------

            if creates_kAP(
                current_set,
                x,
                k,
                depth_stats[depth],
                min_value
            ):

                stats["ap_prunes"] += 1
                depth_stats[depth]["ap_prunes"] += 1

                continue

            # -----------------------------------------
            # LOOK-AHEAD PRUNE
            #
            # Pretend we choose x, then count how many
            # remaining values are individually still
            # eligible.
            # -----------------------------------------

            test_set = current_set.copy()
            test_set.add(x)

            needed_after_x = size - (len(current) + 1)

            eligible_after_x = 0

            for j in range(i + 1, len(S)):

                y = S[j]

                if not creates_kAP(
                    test_set,
                    y,
                    k,
                    depth_stats[depth],
                    min_value
                ):
                    eligible_after_x += 1

            if eligible_after_x < needed_after_x:

                stats["lookahead_prunes"] += 1
                depth_stats[depth]["lookahead_prunes"] += 1

                continue

            # -----------------------------------------
            # x SURVIVES
            # -----------------------------------------

            stats["choices_survived"] += 1
            depth_stats[depth]["choices_survived"] += 1

            # Choose
            current.append(x)
            current_set.add(x)

            # Explore
            yield from backtrack(
                i + 1,
                current,
                current_set
            )

            # Undo
            current.pop()
            current_set.remove(x)

    yield from backtrack(
        start=0,
        current=[],
        current_set=set()
    )


def r(N: int, k: int, previous_max: int):
    """
    Compute r_k(N), assuming r_k(N-1) = previous_max.

    Since r_k(N) can increase by at most 1 when N increases by 1,
    the only target we need to test is previous_max + 1.
    """

    stats = {
        "nodes": 0,
        "cache_hits": 0,
        "cache_misses": 0,
        "ap_prunes": 0,
        "lookahead_prunes": 0,
        "insufficient_remaining_prunes": 0,
        "choices_tried": 0,
        "choices_survived": 0,
        "candidates_yielded": 0,
        "elapsed_seconds": 0.0
    }

    depth_stats = make_depth_stats()

    target_size = previous_max + 1

    S = range(1, N + 1)

    start_time = perf_counter()


    for candidate in generate_AP_free_candidates(
        S,
        target_size,
        k,
        stats,
        depth_stats
    ):

        stats["elapsed_seconds"] = (
            perf_counter() - start_time
        )

        return (
            target_size,
            candidate,
            stats,
            depth_stats
        )


    stats["elapsed_seconds"] = (
        perf_counter() - start_time
    )

    return (
        previous_max,
        None,
        stats,
        depth_stats
    )


In [4]:
def print_depth_stats(depth_stats):

    print()
    max_node_depth = max(
        depth_stats,
        key=lambda depth: depth_stats[depth]["nodes"]
    )

    print(f"{max_node_depth} is max node depth.")
    print(
        f"{'Depth':>25} | "
        f"{'Nodes':>12} | "
        f"{'Cache Hits':>12} | "
        f"{'Cache Misses':>12} | "
        f"{'Short Size Prunes':>17} | "
        f"{'Choices Tried':>14} | "
        f"{'AP Prunes':>12} | "
        f"{'LA Prunes':>12} | "
        f"{'Choices Survived':>17} | "
        f"{'Yielded':>8}"
    )

    print(" " * 15 + "-" * 110)

    for depth in sorted(depth_stats):

        row = depth_stats[depth]

        print(
            f"{depth:>25} | "
            f"{row['nodes']:>12,} | "
            f"{row['cache_hits']:>12,} | "
            f"{row['cache_misses']:>12,} | "
            f"{row['remain_prunes']:>17,} | "
            f"{row['choices_tried']:>14,} | "
            f"{row['ap_prunes']:>12,} | "
            f"{row['lookahead_prunes']:>12,} | "
            f"{row['choices_survived']:>17,} | "
            f"{row['candidates_yielded']:>8,}"
        )

In [ ]:
# --------------------------------------------------
# Experiment settings
# --------------------------------------------------

N_start = 5
N_end = 70

k = 3

# Known starting value:
# r_3(5) = 4

max_size = 4

results = []


# Starting known value

print(
    f"{N_start:>3} | "
    f"{max_size:>6} | "
    f"{0:>10.4f} | "
    f"{0:>12} | "
    f"{0:>12} | "
    f"{0:>14} | "
    f"{0:>12} | "
    f"{0:>12} | "
    f"{0:>8}"
)

results.append(
    (N_start, max_size)
)


# --------------------------------------------------
# Main experiment
# --------------------------------------------------

for N in range(
    N_start + 1,
    N_end + 1
):

    # Save what size we are trying to find
    target_size = max_size + 1

    (
        max_size,
        candidate,
        stats,
        depth_stats
    ) = r(
        N,
        k,
        max_size
    )

    print()
    print(f"RESULT FOR N = {N}")
    print("-" * 120)

    print(
        f"{'N':>3} | "
        f"{'r_k(N)':>6} | "
        f"{'Seconds':>10} | "
        f"{'Nodes':>12} | "
        f"{'Cache Hits':>12} | "
        f"{'Cache Misses':>12} | "
        f"{'Short Size Prunes':>17} | "
        f"{'Tried':>14} | "
        f"{'AP Prunes':>12} |"
        f"{'LA Prunes':>12} | "
        f"{'Survived':>12} | "
        f"{'Yielded':>8}"
    )

    print("-" * 120)

    print(
        f"{N:>3} | "
        f"{max_size:>6} | "
        f"{stats['elapsed_seconds']:>10.4f} | "
        f"{stats['nodes']:>12,} | "
        f"{stats['cache_hits']:>12,} | "
        f"{stats['cache_misses']:>12,} | "
        f"{stats['insufficient_remaining_prunes']:>17,} | "
        f"{stats['choices_tried']:>14,} | "
        f"{stats['ap_prunes']:>12,} | "
        f"{stats['lookahead_prunes']:>12,} | "
        f"{stats['choices_survived']:>12,} | "
        f"{stats['candidates_yielded']:>8,}"
    )

    results.append(
        (N, max_size)
    )
        
    print()
    print(" " * 15 +
        f"Depth statistics for N = {N}, "
        f"searching for subset size {target_size}"
    )

    print_depth_stats(
        depth_stats
    )

  5 |      4 |     0.0000 |            0 |            0 |              0 |            0 |            0 |        0

RESULT FOR N = 6
------------------------------------------------------------------------------------------------------------------------
  N | r_k(N) |    Seconds |        Nodes |   Cache Hits | Cache Misses | Short Size Prunes |          Tried |    AP Prunes |   LA Prunes |     Survived |  Yielded
------------------------------------------------------------------------------------------------------------------------
  6 |      4 |     0.0001 |            4 |            0 |            0 |                 0 |             19 |            1 |           15 |            3 |        0

               Depth statistics for N = 6, searching for subset size 5

1 is max node depth.
                    Depth |        Nodes |   Cache Hits | Cache Misses | Short Size Prunes |  Choices Tried |    AP Prunes |    LA Prunes |  Choices Survived |  Yielded
               ---------------------

In [ ]:
current = (1,2,3)
x = 4
key = (frozenset(current),x)
cache = {}
cache[key] = True

if key in cache:
    print("Key is in cache")

Key is in cache
